# B1–B2: Remittance Orchestration, MCP and A2A Integration Lab

Scenario: an accounts-receivable team ingests remittance advice, validates it, performs deterministic and fuzzy payment matching, then escalates exceptions to the Azure AI Foundry agent.

In [1]:
from difflib import SequenceMatcher

INVOICES = [
    {'invoice_id': 'INV-1001', 'customer': 'Acme Retail', 'amount': 12500.00},
    {'invoice_id': 'INV-1002', 'customer': 'Northwind Traders', 'amount': 8300.00},
]
REMITTANCES = [
    {'payment_id': 'PAY-77', 'reference': 'INV-1001', 'payer': 'Acme Retail', 'amount': 12500.00},
    {'payment_id': 'PAY-78', 'reference': 'INV-100X', 'payer': 'Northwind Trading', 'amount': 8300.00},
]

# B2: tool contracts an orchestrator can expose through MCP or call through A2A.
MCP_TOOLS = {
    'get_invoice': {'input': {'invoice_id': 'string'}, 'output': 'invoice | null'},
    'post_cash_receipt': {'input': {'payment_id': 'string', 'invoice_id': 'string'}, 'output': 'receipt_id'},
}
A2A_AGENT_CARD = {'name': 'MatchingAgent', 'skills': ['two_way_match', 'fuzzy_match', 'exception_routing']}

def validate_remittance(payment):
    required = {'payment_id', 'reference', 'payer', 'amount'}
    return required.issubset(payment) and payment['amount'] > 0

def two_way_match(payment, invoices):
    return next((i for i in invoices if i['invoice_id'] == payment['reference'] and i['amount'] == payment['amount']), None)

def fuzzy_match(payment, invoices, threshold=0.85):
    ranked = []
    for invoice in invoices:
        name_score = SequenceMatcher(None, payment['payer'].lower(), invoice['customer'].lower()).ratio()
        amount_score = 1.0 if payment['amount'] == invoice['amount'] else 0.0
        score = 0.65 * name_score + 0.35 * amount_score
        ranked.append((score, invoice))
    score, invoice = max(ranked, key=lambda x: x[0])
    return invoice if score >= threshold else None, round(score, 2)

def run_supervisor(payments):
    results = []
    for payment in payments:
        if not validate_remittance(payment):
            results.append({'payment_id': payment.get('payment_id'), 'status': 'rejected', 'reason': 'invalid format'})
            continue
        exact = two_way_match(payment, INVOICES)
        if exact:
            results.append({'payment_id': payment['payment_id'], 'status': 'auto_matched', 'invoice_id': exact['invoice_id']})
            continue
        candidate, confidence = fuzzy_match(payment, INVOICES)
        results.append({'payment_id': payment['payment_id'], 'status': 'human_review', 'candidate': candidate['invoice_id'] if candidate else None, 'confidence': confidence})
    return results

matching_results = run_supervisor(REMITTANCES)
print('MCP tools:', list(MCP_TOOLS))
print('A2A agent card:', A2A_AGENT_CARD['name'])
matching_results

MCP tools: ['get_invoice', 'post_cash_receipt']
A2A agent card: MatchingAgent


[{'payment_id': 'PAY-77', 'status': 'auto_matched', 'invoice_id': 'INV-1001'},
 {'payment_id': 'PAY-78',
  'status': 'human_review',
  'candidate': 'INV-1002',
  'confidence': 0.89}]

In [2]:
# B1: the supervisor delegates only the exception to the deployed Foundry agent.
import os, requests
from dotenv import load_dotenv
load_dotenv('../.env')

def output_text(response_json):
    return next(part['text'] for item in response_json['output'] if item.get('type') == 'message' for part in item['content'] if part.get('type') == 'output_text')

exception = next(item for item in matching_results if item['status'] == 'human_review')
prompt = f'''You are an accounts-receivable exception analyst. A payment matching workflow produced: {exception}.
Give a concise recommendation with: probable invoice, evidence, next action, and whether human approval is required.
Do not authorize posting a payment automatically when the reference differs.'''
response = requests.post(
    os.environ['AGENT_ENDPOINT'],
    params={'api-version': 'v1'},
    headers={'api-key': os.environ['AZURE_OPENAI_API_KEY'], 'Content-Type': 'application/json'},
    json={'input': prompt, 'stream': False}, timeout=120,
)
response.raise_for_status()
print(output_text(response.json()))

Recommendation for PAY-78

- Probable invoice: INV-1002
- Evidence: System’s top candidate with 0.89 confidence; routed to “human_review,” indicating ambiguity. Per control, no auto-posting when the payment reference does not explicitly match an invoice number.
- Next action:
  1) Pull remittance/statement for PAY-78; confirm INV-1002 is referenced.
  2) Cross-check payer, currency, and value date; verify payment amount equals the open balance on INV-1002 and that INV-1002 is still open in ERP; rule out other same-amount open invoices for the payer.
  3) If all checks pass, prepare application to INV-1002 and attach evidence; if any mismatch or uncertainty, seek customer confirmation or park as unapplied.
- Human approval required: Yes (status = human_review; do not auto-post when references differ).
